# Ordered Logistic Regression Results Dataset Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset—*Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya*—using the `mlcroissant` library.

### Dataset Source
This dataset is described using a Croissant schema and is available from the following URL:

[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Install mlcroissant if not present. Restart the kernel after installing if needed.
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and records using the `mlcroissant` library. We'll inspect the dataset's description and core properties to understand its context.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# View and print core metadata fields
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Cite as: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Let's discover the available **Record Sets** in the package, as well as their **Fields** and associated `@id`s. We'll use these `@id`s for all future data access and filtering steps.

Below, we print the available record sets and describe their basic structural elements using their unique Croissant `@id` fields.

In [ ]:
print("Record sets in this dataset:")
for rs in dataset.record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '(no name)')}")
    # List fields for each record set
    if 'field' in rs:
        if isinstance(rs['field'], dict):
            fields = [rs['field']]
        else:
            fields = rs['field']
        print(f"  Fields:")
        for ff in fields:
            print(f"    - @id: {ff['@id']}, name: {ff.get('name','')} (type: {ff.get('dataType','')})")
    else:
        print("  (No fields found)")
    print('')
if not dataset.record_sets:
    print("No record sets defined in the dataset. If this occurs, check the Croissant schema for available record sets.")

Let's enumerate any available records for each record set, referencing them by their `@id`. (If there are no record sets, this loop will have no output.)

In [ ]:
# For demonstration, try to print the first few records for each record set by @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets] if hasattr(dataset, 'record_sets') and dataset.record_sets else []
for rsid in record_set_ids:
    print(f'\nRecords for RecordSet @id = {rsid}:')
    try:
        for i, record in enumerate(dataset.records(record_set=rsid)):
            print(record)
            if i >= 2:
                print('...')
                break
    except Exception as e:
        print(f"Could not load records for {rsid}: {e}")
if not record_set_ids:
    print("No record sets available to load records.")

## 3. Data Extraction
Next, we'll load the data from all available record sets into pandas DataFrames, referencing them exclusively by their Croissant `@id`. This enables flexible downstream access and manipulation.

In [ ]:
dataframes = {}
if record_set_ids:
    for rsid in record_set_ids:
        try:
            records = list(dataset.records(record_set=rsid))
            if records:
                df = pd.DataFrame(records)
                dataframes[rsid] = df
                print(f"Loaded {len(df)} records for RecordSet @id {rsid}.")
            else:
                print(f"No records found for RecordSet @id {rsid}.")
        except Exception as e:
            print(f"Error loading RecordSet {rsid}: {e}")
    # Display columns of the first dataframe if any
    if dataframes:
        first_id = next(iter(dataframes.keys()))
        print(f"\nColumns available in first record set ({first_id}):")
        print(dataframes[first_id].columns.tolist())
        display(dataframes[first_id].head())
    else:
        print("No dataframes available for inspection.")
else:
    print("No record sets identified; cannot extract data.")

## 4. Exploratory Data Analysis (EDA)
Let's apply basic EDA steps to one record set (if present). We'll:
1. Filter rows using a numeric field (using its `@id`).
2. Normalize the selected numeric field (z-score).
3. Group by a categorical field and take the mean of the numeric field.

We will automatically choose the first record set and suitable fields if available. **All field accesses reference columns by their Croissant `@id`.**

In [ ]:
if dataframes:
    first_rs = next(iter(dataframes.keys()))
    df = dataframes[first_rs]
    print(f"Working with record set @id: {first_rs}")
    
    # Heuristic: select first integer/float-like column as numeric, and first object/categorical for grouping
    numeric_col = None
    group_col = None
    for col in df.columns:
        # Try to infer numeric/int field
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_col = col
            break
    for col in df.columns:
        if (not pd.api.types.is_numeric_dtype(df[col])) and (df[col].dtype==object):
            group_col = col
            break
    
    if numeric_col is not None:
        print(f"Using numeric field (by @id): {numeric_col}")
        threshold = df[numeric_col].mean() if pd.notnull(df[numeric_col].mean()) else 0
        filtered_df = df[df[numeric_col] > threshold]
        print(f"Filtered records where {numeric_col} > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Normalize
        normed_col = f"{numeric_col}_normalized"
        filtered_df[normed_col] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
        print(f"Normalized field {numeric_col} (added column {normed_col}):")
        display(filtered_df[[numeric_col, normed_col]].head())
        
        # Group if possible
        if group_col is not None:
            print(f"Grouping by field (by @id): {group_col}")
            grouped = filtered_df.groupby(group_col)[numeric_col].mean().to_frame()
            display(grouped.head())
        else:
            print("No suitable group (categorical) field found.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's produce a histogram of the numeric field, and—where possible—a bar plot of means for different categories of the grouping field. 

> All plots are constructed referencing DataFrame columns as their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    first_rs = next(iter(dataframes.keys()))
    df = dataframes[first_rs]
    numeric_col = None
    group_col = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_col = col
            break
    for col in df.columns:
        if (not pd.api.types.is_numeric_dtype(df[col])) and (df[col].dtype==object):
            group_col = col
            break

    if numeric_col:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_col].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_col} (@id)")
        plt.xlabel(numeric_col)
        plt.ylabel('Count')
        plt.show()

        if group_col:
            plt.figure(figsize=(10,4))
            sns.barplot(
                x=group_col,
                y=numeric_col,
                data=df,
                estimator='mean' if hasattr(sns, 'barplot') else None,
                ci=None
            )
            plt.title(f"Mean of {numeric_col} grouped by {group_col} (by @id)")
            plt.xlabel(group_col)
            plt.ylabel(f"Mean {numeric_col}")
            plt.xticks(rotation=30, ha='right')
            plt.tight_layout()
            plt.show()
        else:
            print("No suitable group field found for bar plot.")
    else:
        print("No numeric field found for visualization.")
else:
    print("No data available to visualize.")

## 6. Conclusion
This notebook demonstrated how to load, examine, filter, and visualize data from a Croissant-packaged FAIR² dataset using `mlcroissant`.

- All data access was handled through the dataset's entity `@id` fields, ensuring reproducibility and schema-compatibility.
- You can use this notebook as a template for investigating other Croissant-formatted datasets or extending it for domain-specific analyses.

**Next steps:** Review the documentation for [`mlcroissant`](https://pypi.org/project/mlcroissant/) and [MLCommons Croissant](https://mlcommons.org/working-groups/data/croissant/) to learn more about schema design and programmatic data workflows.